# Phase 4 — Bootstrap CIs, AUUC, and calibration

Phase 3 produced six Qini coefficients on the same 2.8M test set, but nothing separates real gaps from sampling noise. This notebook tightens the evaluation with three additions:

1. **Bootstrap 95% CI on the Qini coefficient** (200 resamples).
2. **AUUC** — Area Under the Uplift Curve — a second within-dataset ranking metric.
3. **Calibration** — bin by predicted-uplift decile and compare each bin's mean prediction to its observed treated-vs-control difference. Well-calibrated models sit on the 45° line.

No refits here — models come from the joblib artifacts written at the end of notebook 03 (`../artifacts/models/*.joblib`).

In [ ]:
import sys
import time
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / 'pyproject.toml').exists():
        REPO_ROOT = candidate
        break
sys.path.insert(0, str(REPO_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.uplift.data import load_criteo, FEATURE_COLS
from src.uplift.split import make_split
from src.uplift.evaluation import (
    qini_curve, qini_coefficient, auuc,
    bootstrap_qini_ci, uplift_calibration,
)
from src.uplift.plots import (
    plot_qini_curves_with_ci, plot_uplift_calibration,
)

sns.set_theme(style='whitegrid', context='notebook', palette='deep')

ARTIFACTS = REPO_ROOT / 'artifacts'
MODELS_DIR = ARTIFACTS / 'models'
ARTIFACTS.mkdir(exist_ok=True)

## 1. Load the shared test set

Same 80/20 split as Phases 2 and 3 (seed 42, stratified on treatment). The bootstrap needs the raw test arrays, so they get materialized once.

In [ ]:
df = load_criteo()
_, test = make_split(df)

X_te = test[FEATURE_COLS].to_numpy()
T_te = test['treatment'].to_numpy()
Y_te = test['conversion'].to_numpy()
print(f'test set: {X_te.shape[0]:,} rows × {X_te.shape[1]} features')

## 2. Load persisted models

Each `.joblib` payload is a dict: `{model, train_size, fit_time_s}`.

In [ ]:
model_files = sorted(MODELS_DIR.glob('*.joblib'))
assert model_files, (
    f'no persisted models found in {MODELS_DIR}. '
    'Run notebook 03 first (its last cell writes them).'
)

MODEL_ORDER = [
    'Propensity (baseline)',
    'S-learner',
    'T-learner',
    'Class Transformation',
    'X-learner',
    'Causal Forest',
]

models = {}
train_sizes = {}
fit_times = {}
for p in model_files:
    payload = joblib.load(p)
    name = p.stem
    models[name] = payload['model']
    train_sizes[name] = payload['train_size']
    fit_times[name] = payload['fit_time_s']

# Reorder to the canonical presentation order
models = {n: models[n] for n in MODEL_ORDER if n in models}
print('loaded:', list(models.keys()))

## 3. Predict uplift on the test set

One pass per model, cached so the downstream cells (Qini, AUUC, bootstrap, calibration) don't have to recompute.

In [ ]:
uplift_preds = {}
for name, m in models.items():
    t0 = time.time()
    uplift_preds[name] = m.predict_uplift(X_te)
    print(f'  {name:<25s} predicted in {time.time()-t0:5.1f}s')

## 4. Qini coefficient, AUUC, and bootstrap 95% CI

**Bootstrap implementation.** 200 resamples of the 2.8M test rows with replacement. `bootstrap_qini_ci` uses a multiplicity trick: sort by predicted uplift once, then per resample draw counts via `bincount(rng.integers)` and use weighted cumulative sums. Mathematically identical to naive index resampling (verified against it seed-for-seed) but roughly 4× faster because the argsort happens once instead of per iteration.

In [ ]:
N_BOOT = 200
SEED = 42

curves = {}
coefs = {}
auucs = {}
cis = {}

for name, u in uplift_preds.items():
    curve = qini_curve(Y_te, T_te, u)
    curves[name] = curve
    coefs[name] = qini_coefficient(curve)
    auucs[name] = auuc(curve)
    t0 = time.time()
    cis[name] = bootstrap_qini_ci(Y_te, T_te, u, n_boot=N_BOOT, seed=SEED)
    print(f'  {name:<25s} Q={coefs[name]:+.4f}  CI=[{cis[name]["lo"]:+.4f}, {cis[name]["hi"]:+.4f}]  ({time.time()-t0:.1f}s)')

## 5. Summary table

One row per model, sorted by Qini coefficient (highest first). This is the headline table that lands in the README.

In [ ]:
summary = pd.DataFrame({
    'model': list(models.keys()),
    'train_size': [train_sizes[n] for n in models],
    'fit_time_s': [fit_times[n] for n in models],
    'qini_coefficient': [coefs[n] for n in models],
    'qini_ci_lo': [cis[n]['lo'] for n in models],
    'qini_ci_hi': [cis[n]['hi'] for n in models],
    'auuc': [auucs[n] for n in models],
}).sort_values('qini_coefficient', ascending=False).reset_index(drop=True)

summary.to_csv(ARTIFACTS / 'results.csv', index=False)
print(f'wrote {ARTIFACTS / "results.csv"}')

summary.style.format({
    'fit_time_s': '{:.1f}',
    'qini_coefficient': '{:+.4f}',
    'qini_ci_lo': '{:+.4f}',
    'qini_ci_hi': '{:+.4f}',
    'auuc': '{:+.4f}',
}).bar(subset=['qini_coefficient'], color='#b6532c', align=0)

## 6. Qini curves with bootstrap CI in the legend

Same six-curve layout as Phase 3, but each legend entry now carries `Q = mean [lo, hi]` from the bootstrap. Overlapping CIs between two curves mean the Qini gap between those models isn't statistically distinguishable on this test set.

In [ ]:
ordered = dict(sorted(curves.items(), key=lambda kv: -coefs[kv[0]]))
ordered_ci = {n: cis[n] for n in ordered}

fig, ax = plt.subplots(figsize=(10.5, 6))
plot_qini_curves_with_ci(ordered, ordered_ci, ax=ax)
fig.tight_layout()
fig.savefig(ARTIFACTS / 'qini_with_ci.png', dpi=140, bbox_inches='tight')
plt.show()

## 7. Calibration — predicted vs observed uplift by decile

Each model's predictions get sorted into 10 quantile bins. Within a bin, observed uplift is `mean(Y | T=1) − mean(Y | T=0)`. Randomization identifies this bin-level ATE without adjustment — random assignment stays random inside any partition of X that ignores post-treatment information.

Points on the 45° line = calibrated. Systematically above the line = model underpredicts. Below the line = overpredicts. A model can rank well (good Qini) while being badly miscalibrated in level — the two metrics answer different questions. Ranking says who to target; calibration says whether the predicted uplift is a trustworthy decision variable for anything downstream (budget allocation, expected-value math).

In [ ]:
cal_tables = {
    name: uplift_calibration(Y_te, T_te, u, n_bins=10)
    for name, u in uplift_preds.items()
}
cal_tables['S-learner']

## 8. Calibration grid — all six models

Error bars are the two-sample-Bernoulli SE for the difference in means inside each bin.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, name in zip(axes.flat, models.keys()):
    plot_uplift_calibration(cal_tables[name], ax=ax, title=name)
fig.suptitle('Uplift calibration — predicted vs observed, by decile of predicted uplift',
             y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(ARTIFACTS / 'calibration_grid.png', dpi=140, bbox_inches='tight')
plt.show()

## 9. Reading the results

**CI table.** A model whose CI overlaps zero isn't distinguishable from random targeting on this test set. Two models whose CIs overlap each other are tied — the point-estimate gap sits inside sampling noise.

**Qini vs AUUC.** Both reward finding persuadables at the top of the ranking and usually agree on the ordering. When they diverge, AUUC (raw area under Q(k)) is more sensitive to whole-curve shape, while the Qini coefficient (area vs the random line) leans on the top of the ranking. For the writeup: lead with Qini + CI, mention AUUC as a robustness check.

**Calibration is orthogonal to ranking.** A model can dominate on Qini while being 2× off in level — predicted uplift inflated but ordering correct. Ranking metrics answer "who to target." Calibration answers "is the predicted uplift number itself usable" for anything downstream.

**Model-specific priors.** The propensity baseline's "predicted uplift" is actually a conversion probability — expect its calibration plot to look wildly off in level even where the ranking has some signal for high-baseline users. The S-learner is the textbook failure candidate: if HGB never splits on the T column, `μ(x,1) − μ(x,0) ≡ 0` and the calibration panel shows "no calibration signal (constant uplift prediction)" — the treatment feature has been fully swallowed by regularization on a large feature set.